# NHS A&E Waiting Times — SQL Analysis

This notebook repeats and extends the Python analysis using **SQL**. The cleaned trust-level data (`ae_clean_all_years.csv`) is loaded into a SQLite database, and every query lives in its own file in the `sql/` folder so it can be read, reused or run in any SQL tool.

The SQL layer does four things:

1. **Validates** the data before trusting it.
2. **Reproduces** the national and regional results from the Python notebook, as a cross-check.
3. **Extends** the analysis with patient-weighted averages, year-on-year change and rankings, using window functions (`RANK`, `LAG`, `FIRST_VALUE`, `NTILE`).
4. **Answers** the question of which trusts *consistently* miss the target, across all five years.

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

con = sqlite3.connect(":memory:")
con.executescript(Path("sql/01_schema.sql").read_text())

data = pd.read_csv("ae_clean_all_years.csv")
data.to_sql("ae_raw", con, if_exists="append", index=False)
print(f"Loaded {len(data)} rows into ae_raw")


def run(sql_file, statement=None):
    """Run a .sql file. Returns a DataFrame for the chosen SELECT statement."""
    text = Path(sql_file).read_text()
    statements, buf = [], []
    for line in text.splitlines():
        buf.append(line)
        if sqlite3.complete_statement("\n".join(buf)):
            stmt = "\n".join(buf).strip()
            if any(l.strip() and not l.strip().startswith("--") for l in stmt.splitlines()):
                statements.append(stmt)
            buf = []
    selects = [s for s in statements if "SELECT" in s.upper() and "CREATE" not in s.upper()]
    for s in statements:
        if s not in selects:
            con.executescript(s)
    if selects:
        return pd.read_sql(selects[statement or 0], con)

Loaded 623 rows into ae_raw


## 1. Data quality checks

Before analysing, check the data. Two issues were found in the cleaned file:

- **National "Total" rows** were meant to be removed in Python, but four slipped through (the Python filter did not catch them). Left in, each would be counted as an extra trust.
- **Region names carry trailing spaces** (for example `"North West "`), which can silently split one region into two when grouping.

The remaining checks confirm totals, percentages and uniqueness are all correct.

In [2]:
for i, title in enumerate([
    "Rows per snapshot",
    "Summary 'Total' rows that should not be there",
    "Region names with stray spaces",
    "Totals that don't add up (should be 0)",
    "Percentages that don't recalculate (should be 0)",
    "Duplicate trust-month rows (should be empty)",
    "Impossible values (should be 0)",
]):
    print(f"\n{i + 1}. {title}")
    display(run("sql/02_data_quality_checks.sql", i))


1. Rows per snapshot


,month,rows
0,March 2021,128
1,March 2022,126
2,March 2023,125
3,March 2024,122
4,March 2025,122



2. Summary 'Total' rows that should not be there


,month,trust,total_attendances,pct_within_4hrs
0,March 2021,TOTAL,1622302,87.3
1,March 2022,TOTAL,2100036,74.2
2,March 2023,TOTAL,2093221,74.0
3,March 2025,TOTAL,2299660,74.4



3. Region names with stray spaces


...

,bad_percentages
0,0



6. Duplicate trust-month rows (should be empty)


,month,trust,copies



7. Impossible values (should be 0)


,impossible_rows
0,0


## 2. A clean view

Both problems are fixed once, in a view (`sql/03_clean_view.sql`), which every later query reads from: region names are trimmed, "Total" rows are excluded, and only major A&E providers (at least 100 Type 1 attendances) are kept.

In [3]:
run("sql/03_clean_view.sql")
pd.read_sql("SELECT COUNT(*) AS trust_year_records FROM ae", con)

,trust_year_records
0,619


## 3. National trend

The simple average matches the Python notebook exactly, which confirms both methods agree. The **patient-weighted** column counts every patient equally rather than every trust, one of the planned extensions in the original project.

In [4]:
run("sql/04_national_trend.sql")

,year,trusts,attendances,avg_pct_within_4hrs,weighted_pct_within_4hrs,gap_to_95pct_target
0,2021,127,1489268,86.6,86.2,8.8
1,2022,125,1903323,71.7,71.8,23.2
2,2023,124,1872378,70.9,71.3,23.7
3,2024,122,2067604,70.9,71.4,23.6
4,2025,121,2075357,71.4,72.0,23.0


## 4. Regional performance

Ranked within each year, worst first, using the patient-weighted figure.

In [5]:
run("sql/05_regional_performance.sql")

,year,region,trusts,avg_pct_within_4hrs,weighted_pct_within_4hrs,rank_worst_first
0,2021,North West,20,82.2,81.6,1
1,2021,Midlands,21,84.9,82.8,2
2,2021,South West,15,86.4,86.6,3
3,2021,North East And Yorkshire,22,87.1,86.9,4
4,2021,East Of England,13,87.6,88.3,5
5,2021,London,18,89.4,89.2,6
6,2021,South East,18,89.4,89.3,7
7,2022,North West,19,65.1,65.1,1
8,2022,Midlands,21,71.1,68.8,2
9,2022,South East,17,72.4,72.2,3


## 5. Worst-performing trusts in March 2025

`type1_pct_within_4hrs` shows performance in major A&E departments alone. For some trusts it is much lower than the all-types figure, because their minor-injury units lift the overall number.

In [6]:
run("sql/06_worst_trusts.sql")

,rank_in_year,trust,region,total_attendances,pct_within_4hrs,type1_pct_within_4hrs
0,1,EAST CHESHIRE NHS TRUST,North West,4450,50.0,50.0
1,2,THE SHREWSBURY AND TELFORD HOSPITAL NHS TRUST,Midlands,13984,52.5,42.8
2,3,NOTTINGHAM UNIVERSITY HOSPITALS NHS TRUST,Midlands,15958,53.0,46.4
3,4,AIREDALE NHS FOUNDATION TRUST,North East And Yorkshire,6124,58.5,56.9
4,4,HULL UNIVERSITY TEACHING HOSPITALS NHS TRUST,North East And Yorkshire,14477,58.5,41.3
5,4,ROYAL UNITED HOSPITALS BATH NHS FOUNDATION TRUST,South West,8762,58.5,58.5


## 6. Year-on-year change by region

`LAG` compares each year with the one before, and `FIRST_VALUE` measures the total change since 2021.

In [7]:
run("sql/07_year_on_year_change.sql")

,region,year,pct_within_4hrs,change_vs_prior_year,change_since_2021
0,East Of England,2021,88.3,NaN,0.0
1,East Of England,2022,73.0,-15.4,-15.4
2,East Of England,2023,73.7,0.8,-14.6
3,East Of England,2024,74.4,0.6,-14.0
4,East Of England,2025,72.5,-1.8,-15.8
5,London,2021,89.2,NaN,0.0
6,London,2022,74.8,-14.4,-14.4
7,London,2023,72.0,-2.8,-17.2
8,London,2024,73.9,1.8,-15.4
9,London,2025,75.5,1.6,-13.8


## 7. Trusts that consistently miss the target

A single year can be noisy. This query uses `NTILE(5)` to place every trust into a national quintile each year, then keeps trusts in the **bottom fifth in at least four of the five years**. Seven trusts were in the bottom fifth every single year.

In [8]:
run("sql/08_persistent_underperformers.sql")

,trust,region,years_reported,years_in_bottom_fifth,avg_pct_within_4hrs,worst_year_pct
0,THE SHREWSBURY AND TELFORD HOSPITAL NHS TRUST,Midlands,5,5,56.0,49.0
1,HULL UNIVERSITY TEACHING HOSPITALS NHS TRUST,North East And Yorkshire,5,5,56.8,44.9
2,EAST CHESHIRE NHS TRUST,North West,5,5,57.9,50.0
3,UNIVERSITY HOSPITALS BIRMINGHAM NHS FOUNDATION...,Midlands,5,5,58.6,50.7
4,UNIVERSITY HOSPITALS OF LEICESTER NHS TRUST,Midlands,5,5,60.0,55.3
5,GLOUCESTERSHIRE HOSPITALS NHS FOUNDATION TRUST,South West,5,5,61.8,56.8
6,MID CHESHIRE HOSPITALS NHS FOUNDATION TRUST,North West,5,5,62.6,55.8
7,THE PRINCESS ALEXANDRA HOSPITAL NHS TRUST,East Of England,5,4,61.5,46.1
8,ROYAL UNITED HOSPITALS BATH NHS FOUNDATION TRUST,South West,5,4,62.1,52.0
9,UNITED LINCOLNSHIRE HOSPITALS NHS TRUST,Midlands,4,4,62.8,57.0


## Summary

- The SQL reproduces the Python results exactly for the national trend and regional averages.
- Data checks caught national "Total" rows and untrimmed region names in the cleaned file; the clean view corrects both.
- Patient-weighted national performance was **72.0%** in March 2025, **23 percentage points** below the 95% standard.
- **Seven trusts** sat in the national bottom fifth in all five years, led by The Shrewsbury and Telford Hospital (five-year average 56.0%) and Hull University Teaching Hospitals (56.8%).